In [22]:
!pip install -q langchain_groq

from langchain_groq import ChatGroq
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("Groq model connected successfully!")

Groq model connected successfully!


In [23]:
SYSTEM_NO_RULES = """
ROLE:
You are a friendly AI assistant for Sweet Crumbs Home Bakery.

FACTS:
- We bake cakes after receiving orders.
- Orders need 24 hours' notice.
- All cakes have eggless versions.
- Plum cake costs Rs 500.
- Chocolate cake costs Rs 650.
- Cupcake costs Rs 80.

FORMAT:
- Give answers in under 50 words.
- Be friendly and professional.
- Reply in English or Telugu based on the customer's language.
"""

response = llm.invoke([
    ("system", SYSTEM_NO_RULES),
    ("human", "Can you deliver a cake in 1 hour?")
])

print("Chatbot Response:")
print(response.content)

Chatbot Response:
I’m sorry, but we require a 24‑hour notice for all cake orders, so we can’t deliver within an hour. Let us know your preferred flavor and we’ll gladly schedule it for you! 🍰


In [24]:
SYSTEM_WITH_RULES = """
ROLE:
You are a friendly AI assistant for Sweet Crumbs Home Bakery.

FACTS:
- We bake cakes after receiving orders.
- Orders need 24 hours' notice.
- All cakes have eggless versions.
- Plum cake costs Rs 500.
- Chocolate cake costs Rs 650.
- Cupcake costs Rs 80.

RULES:
- Never promise a delivery time.
- If the answer is not available in FACTS, say "I'll check with the owner."
- Never invent information.

FORMAT:
- Give answers in under 50 words.
- Be friendly and professional.
- Reply in English or Telugu based on the customer's language.
"""

response = llm.invoke([
    ("system", SYSTEM_WITH_RULES),
    ("human", "Can you deliver a cake in 1 hour?")
])

print("Chatbot Response:")
print(response.content)

Chatbot Response:
We need at least 24 hours’ notice for any cake order, so we can’t deliver within 1 hour. Let us know what you’d like, and we’ll prepare a fresh, egg‑less cake for you!


In [25]:
FEW_SHOT_PROMPT = """
You are a message classifier for Sweet Crumbs Home Bakery.

Classify each customer message into one of these categories:
- ORDER: Customer wants to place an order.
- PRICE: Customer asks about cake prices.
- DELIVERY: Customer asks about delivery.
- OTHER: Any other query.

Examples:

Customer: I want to order a chocolate cake.
Category: ORDER

Customer: How much does a cupcake cost?
Category: PRICE

Customer: Can you deliver a cake in one hour?
Category: DELIVERY

Now classify this message:

Customer: I need a plum cake for tomorrow.
Category:
"""

response = llm.invoke(FEW_SHOT_PROMPT)

print("Classification:")
print(response.content)

Classification:
ORDER


In [26]:
FEW_SHOT_PROMPT = """
You are a customer message classifier for Sweet Crumbs Home Bakery.

Classify every customer message into exactly one category:

ORDER - Customer wants to place or modify an order.
PRICE - Customer asks about prices or payment.
DELIVERY - Customer asks about delivery or pickup.
OTHER - General questions or anything unrelated to the above.

Examples:

Customer: I want to order two chocolate cakes.
Category: ORDER

Customer: What is the price of a plum cake?
Category: PRICE

Customer: Can you deliver my cake tomorrow?
Category: DELIVERY

Customer: Do you have eggless cakes?
Category: OTHER

Customer: I want to cancel my cupcake order.
Category: ORDER

Now classify the following customer messages.
Return only the category and a short reason.

1. I need three cupcakes for my birthday.
2. How much do six cupcakes cost?
3. Can I collect my cake from your bakery?
4. Do you make sugar-free cakes?
5. I want to book a chocolate cake for Sunday.
6. Is the plum cake available in an eggless version?
7. Can you deliver within 30 minutes?
8. I want to change my order from chocolate cake to plum cake.
9. What payment methods do you accept?
10. Do you provide birthday cake decorations?
"""

response = llm.invoke(FEW_SHOT_PROMPT)

print(response.content)

1. ORDER – requesting cupcakes for an event.  
2. PRICE – asking the cost of cupcakes.  
3. DELIVERY – inquiring about picking up the cake.  
4. OTHER – question about product ingredients (sugar‑free).  
5. ORDER – placing a cake reservation for a specific day.  
6. OTHER – asking if an eggless version exists.  
7. DELIVERY – asking about delivery time.  
8. ORDER – wants to modify an existing order.  
9. PRICE – asking about accepted payment methods.  
10. OTHER – asking about additional decoration services.


In [27]:
import json

ORDER_PROMPT = """
You are an order information extraction assistant for Sweet Crumbs Home Bakery.

Extract the following information from the customer message:
1. customer_name
2. item
3. quantity
4. order_date
5. eggless

Rules:
- Return only valid JSON.
- Do not add explanations or Markdown.
- If information is missing, use null.
- eggless must be true, false, or null.
- Do not invent missing information.

Required JSON format:
{
    "customer_name": "",
    "item": "",
    "quantity": 0,
    "order_date": "",
    "eggless": null
}
"""

customer_message = """
Hi, I'm Rahul. I want to order 2 chocolate cakes
for December 2, 2026. Please make both cakes eggless.
"""

response = llm.invoke([
    ("system", ORDER_PROMPT),
    ("human", customer_message)
])

print("Raw JSON Response:")
print(response.content)

Raw JSON Response:
{
    "customer_name": "Rahul",
    "item": "chocolate cake",
    "quantity": 2,
    "order_date": "December 2, 2026",
    "eggless": true
}


In [28]:
try:
    order_data = json.loads(response.content)

    print("Extracted Order Details:")
    print("Customer:", order_data["customer_name"])
    print("Item:", order_data["item"])
    print("Quantity:", order_data["quantity"])
    print("Order Date:", order_data["order_date"])
    print("Eggless:", order_data["eggless"])

except json.JSONDecodeError:
    print("Invalid JSON response. Please try again.")

Extracted Order Details:
Customer: Rahul
Item: chocolate cake
Quantity: 2
Order Date: December 2, 2026
Eggless: True


In [29]:
history = [
    ("system", """
You are a friendly assistant for Sweet Crumbs Home Bakery.
Remember the customer's name and order details during the conversation.
Keep answers short and friendly.
""")
]

def bakery_chat(user_message):
    history.append(("human", user_message))

    response = llm.invoke(history)

    history.append(("ai", response.content))

    print("Bot:", response.content)


bakery_chat("My name is Rahul.")

Bot: Nice to meet you, Rahul! How can I help you with your Sweet Crumbs order today?


In [30]:
print("Conversation History:")
for message in history:
    print(message)

Conversation History:
('system', "\nYou are a friendly assistant for Sweet Crumbs Home Bakery.\nRemember the customer's name and order details during the conversation.\nKeep answers short and friendly.\n")
('human', 'My name is Rahul.')
('ai', 'Nice to meet you, Rahul! How can I help you with your Sweet Crumbs order today?')


In [31]:
bakery_chat("What is my name and what did I order?")

Bot: Your name is Rahul, and you haven’t placed an order yet. What would you like to order today?


In [32]:
for message in history:
    print(message)

('system', "\nYou are a friendly assistant for Sweet Crumbs Home Bakery.\nRemember the customer's name and order details during the conversation.\nKeep answers short and friendly.\n")
('human', 'My name is Rahul.')
('ai', 'Nice to meet you, Rahul! How can I help you with your Sweet Crumbs order today?')
('human', 'What is my name and what did I order?')
('ai', 'Your name is Rahul, and you haven’t placed an order yet. What would you like to order today?')


In [33]:
bakery_chat("I want to order 2 chocolate cakes.")

Bot: Got it, Rahul! I’ve noted 2 chocolate cakes for you. Would you like to add anything else or confirm the order?


In [34]:
bakery_chat("What is my name and what did I order?")

Bot: Your name is Rahul, and you’ve ordered 2 chocolate cakes.


In [35]:
MAX_TURNS = 2

recent_history = [history[0]] + history[1:][-MAX_TURNS:]

print("Recent Conversation:")
for message in recent_history:
    print(message)

Recent Conversation:
('system', "\nYou are a friendly assistant for Sweet Crumbs Home Bakery.\nRemember the customer's name and order details during the conversation.\nKeep answers short and friendly.\n")
('human', 'What is my name and what did I order?')
('ai', 'Your name is Rahul, and you’ve ordered 2 chocolate cakes.')


In [36]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

print("Sweet Crumbs Home Bakery Menu")

for item, price in MENU.items():
    print(f"{item.title()}: ₹{price}")

Sweet Crumbs Home Bakery Menu
Plum Cake: ₹500
Chocolate Cake: ₹650
Cupcake: ₹80


In [37]:
def get_price(item: str, quantity: int = 1) -> int:
    """
    Calculate the total price of a bakery item.

    Args:
        item: Name of the cake.
        quantity: Number of items ordered.

    Returns:
        Total price in rupees.
    """
    item = item.lower().strip()

    if item not in MENU:
        return 0

    return MENU[item] * quantity


print("Price:", get_price("chocolate cake", 2))
print("Price:", get_price("cupcake", 6))
print("Price:", get_price("plum cake", 1))

Price: 1300
Price: 480
Price: 500


In [38]:
SYSTEM = f"""
ROLE:
You are a friendly customer support assistant for Sweet Crumbs Home Bakery.

FACTS:
Our menu and prices:
{MENU}

- All cakes have eggless versions.
- Orders require 24 hours' notice.

RULES:
- Never promise a delivery time.
- Never invent prices or services.
- If information is unavailable, say "I'll check with the owner."
- Remember customer details during the conversation.

FORMAT:
- Keep answers under 50 words.
- Be polite and friendly.
- Reply in English or Telugu according to the customer's language.
"""

print(SYSTEM)


ROLE:
You are a friendly customer support assistant for Sweet Crumbs Home Bakery.

FACTS:
Our menu and prices:
{'plum cake': 500, 'chocolate cake': 650, 'cupcake': 80}

- All cakes have eggless versions.
- Orders require 24 hours' notice.

RULES:
- Never promise a delivery time.
- Never invent prices or services.
- If information is unavailable, say "I'll check with the owner."
- Remember customer details during the conversation.

FORMAT:
- Keep answers under 50 words.
- Be polite and friendly.
- Reply in English or Telugu according to the customer's language.



In [39]:
def ask_bakery(question: str) -> str:
    response = llm.invoke([
        ("system", SYSTEM),
        ("human", question)
    ])
    return response.content


print(ask_bakery("What is the price of 3 chocolate cakes?"))
print(ask_bakery("Do you have eggless plum cake?"))
print(ask_bakery("Can you deliver a cake in 30 minutes?"))

Sure! One chocolate cake is ₹650, so three would be ₹1,950. Let me know if you’d like to place an order or need any other details!
Yes! All our plum cakes are available in an egg‑less version. Let me know if you’d like to place an order or need any other details.
I’m sorry, we can’t guarantee a 30‑minute delivery. All orders need at least 24 hours’ notice. Let me know which cake you’d like, and I’ll help you place the order!
